# S13 · Train a neural network to read digits (PyTorch)

This is the portfolio piece. By the end of this notebook you will have trained a
real neural network that reads handwritten digits, and an honest number for how
often it is right. That is the notebook you point to when you say "I trained a
neural network end to end."

We use the same idea as the India Post sorting machine from the README: show the
network thousands of handwritten digits, let it nudge its dials until it reads
them well, then test it on digits it has never seen.

**New here? Read this once.**

- New to Python? You can still run this whole notebook. Press play on each cell,
  top to bottom, and read the note above each one. You will train a real network.
- New to "which way is downhill"? Open `primers/slopes_and_gradients.md`.
- Already confident? The core loop is short; look for the **Stretch (optional)**
  cells near the end for the professional touches (mini-batches, dropout, early
  stopping).
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there. Colab already ships
PyTorch, scikit-learn, NumPy and matplotlib, so there is nothing to install.

In [ ]:
# Everything this notebook uses (torch, scikit-learn, numpy, matplotlib)
# already ships with Colab, so there is nothing to install here.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                       # fast maths on lists of numbers
import matplotlib.pyplot as plt           # drawing charts
from sklearn.datasets import load_digits  # the handwritten-digits dataset
from sklearn.model_selection import train_test_split  # to hold out a test set

np.random.seed(0)

## Step 1 — load the handwritten digits

We use the digits dataset that comes bundled with scikit-learn. Each image is a
tiny 8x8 grid of pixels showing a hand-drawn digit from 0 to 9. It is built in, so
there is nothing to download and it runs offline.

In [ ]:
# Load the data. `images` are 8x8 grids; `data` is the same flattened to 64 numbers.
digits = load_digits()

print("number of images :", digits.images.shape[0])
print("each image shape :", digits.images.shape[1:], "(8 by 8 pixels)")
print("flattened length :", digits.data.shape[1], "numbers per image")
print("first ten labels :", digits.target[:10], "(the true digit for each)")

## Step 2 — look at a few digits

Always look at your data before modelling anything. Here are the first eight
images with their true labels. They are low-resolution, but you can still make out
the digits.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for image, label, ax in zip(digits.images, digits.target, axes.ravel()):
    ax.imshow(image, cmap="gray_r")
    ax.set_title("label: " + str(label))
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## Step 3 — scale the pixels and hold out a test set

Two small chores before training.

First, networks learn best when the inputs are small numbers. The pixels run from
0 to 16, so we divide by 16 to put them between 0 and 1.

Second, we set aside a **test set** the network never sees while learning, so that
later we can judge it honestly on fresh digits, not ones it has memorised.

In [ ]:
# Each image as 64 numbers, scaled to the range 0 to 1.
X = digits.data / 16.0
y = digits.target

# 80% to learn from, 20% kept aside for the final honest test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)

print("training examples:", X_train.shape[0])
print("test examples    :", X_test.shape[0])
print("inputs per image :", X_train.shape[1])

## Step 4 — turn the data into PyTorch tensors

PyTorch works with **tensors**, which are like NumPy arrays but with one extra
power: as numbers flow through the network, PyTorch quietly records how each one
was computed, so it can work out the gradients (which way to nudge every dial) by
itself. That bookkeeping is called **autograd**, and it is the reason we never
hand-derive a gradient again.

We convert our arrays once. The pixel inputs are decimals (`float`); the labels
are whole-number digit classes (`long`).

In [ ]:
import torch

# Set the PyTorch seed so the random starting dials are the same each run.
torch.manual_seed(0)

# Inputs become float tensors; labels become long (integer) tensors.
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

print("X_train tensor shape:", tuple(X_train_t.shape))
print("y_train tensor shape:", tuple(y_train_t.shape))

## Step 5 — build a small network

`nn.Sequential` lets us stack layers in order. Our network:

- a **Linear** layer: 64 inputs → 32 hidden neurons,
- a **ReLU** bend,
- a **Linear** layer: 32 hidden → 10 outputs, one score per digit 0 to 9.

That is the "flexible function with dials" from the README, in three short lines.
The dials are all the numbers inside the two Linear layers, and training will set
them.

In [ ]:
from torch import nn

# 64 inputs -> 32 hidden (with a ReLU bend) -> 10 outputs (one score per digit).
model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10),
)

print(model)

## Step 6 — choose how to measure mistakes, and how to nudge

Two choices before training.

The **loss** is the single "how wrong are we" number. For sorting things into
classes we use **cross-entropy loss**, which is large when the network is confident
and wrong, and small when it is confident and right.

The **optimiser** is what actually nudges the dials downhill using the gradients.
**Adam** is a reliable default. The **learning rate** is the step size.

In [ ]:
# Cross-entropy is the standard loss for sorting into classes.
loss_function = nn.CrossEntropyLoss()

# Adam is a dependable optimiser; lr is the learning rate (step size).
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("Loss and optimiser ready.")

## Step 7 — the training loop

This is the engine of all deep learning, the same handful of steps every time:

1. **forward pass**: send the training digits through, get the network's scores,
2. compute the **loss**: how wrong were those scores,
3. **backward pass**: autograd finds the gradient for every dial (this is backprop),
4. the optimiser **nudges** every dial a little downhill,
5. **repeat** for several **epochs** (one epoch is one full pass over the data),
   watching the loss fall.

We keep it tiny: 30 epochs over the whole training set, which runs in a moment.

In [ ]:
number_of_epochs = 30

# We record the loss each epoch so we can plot it falling.
loss_history = []

for epoch in range(number_of_epochs):
    # 1. Forward pass: the network scores each digit.
    predictions = model(X_train_t)

    # 2. Loss: how wrong are those scores?
    loss = loss_function(predictions, y_train_t)

    # 3. Backward pass: clear old gradients, then backprop the new ones.
    optimizer.zero_grad()
    loss.backward()

    # 4. Nudge every dial a little downhill.
    optimizer.step()

    # Remember this epoch's loss as a plain Python number.
    loss_history.append(loss.item())

    # Print progress every 5 epochs.
    if (epoch + 1) % 5 == 0:
        print("epoch", epoch + 1, " loss:", round(loss.item(), 4))

## Step 8 — watch the loss fall

A healthy training run shows the loss dropping quickly and then flattening out.
That is the network rolling downhill on its "how wrong" landscape, exactly the
picture from the gradients primer.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(loss_history, color="#2E75B6")
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("The loss falls as the network learns")
plt.show()

## Step 9 — the honest test: accuracy on unseen digits

Now the number that matters: how often does the network read the right digit on
images it never saw while learning? We turn off gradient tracking with
`torch.no_grad()` because we are only predicting now, not learning, and that is
faster.

In [ ]:
# torch.no_grad() means "just predict, do not track gradients".
with torch.no_grad():
    test_scores = model(X_test_t)

# For each image, the predicted digit is the one with the highest score.
predicted_digits = torch.argmax(test_scores, dim=1)

# Fraction of test images the network read correctly.
number_correct = (predicted_digits == y_test_t).sum().item()
accuracy = number_correct / len(y_test_t)

print("correct predictions:", number_correct, "out of", len(y_test_t))
print("test accuracy      :", round(accuracy * 100, 2), "%")

## Step 10 — look at a few predictions

Finally, a look at some test images with the digit the network predicted. Green
titles are correct, red titles are mistakes. Seeing the mistakes is a good habit:
the ones it gets wrong are often the ones a human would squint at too.

In [ ]:
# The predicted digits as a NumPy array, for easy plotting.
predicted_digits_numpy = predicted_digits.numpy()

# The test images, reshaped back into 8x8 grids for display.
test_images = X_test.reshape(-1, 8, 8)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for image, predicted, actual, ax in zip(
        test_images, predicted_digits_numpy, y_test, axes.ravel()):
    ax.imshow(image, cmap="gray_r")
    title_color = "green" if predicted == actual else "red"
    ax.set_title("pred: " + str(predicted), color=title_color)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

### Stretch (optional) — the professional way to train

Skip this if the network above already feels like plenty; you have done the real
thing. If you already code, here are three touches that real projects use. All of
them run offline on the same digits.

**Mini-batches with a DataLoader.** Above we fed the whole training set at once.
On big data that will not fit in memory, so instead we serve the data in small,
shuffled **mini-batches**. PyTorch's `DataLoader` does the shuffling and slicing
for us.

**Dropout.** During training, `nn.Dropout` randomly switches off a fraction of the
hidden neurons each step. It stops the network leaning too hard on any one neuron,
which fights **overfitting** (doing well on training data but badly on new data).

**Early stopping.** We watch the loss on a held-out **validation** slice, and stop
training as soon as it stops improving, rather than training on until the network
starts memorising.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# A fresh network, this time with a Dropout layer between the hidden and output.
robust_model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.3),        # randomly drop 30% of hidden neurons each step
    nn.Linear(32, 10),
)

robust_loss = nn.CrossEntropyLoss()
robust_optimizer = torch.optim.Adam(robust_model.parameters(), lr=0.01)

# Carve a small validation slice out of the training data for early stopping.
X_fit_t = X_train_t[:1200]
y_fit_t = y_train_t[:1200]
X_val_t = X_train_t[1200:]
y_val_t = y_train_t[1200:]

# Wrap the fitting data so the DataLoader can serve shuffled mini-batches of 64.
training_dataset = TensorDataset(X_fit_t, y_fit_t)
batch_loader = DataLoader(training_dataset, batch_size=64, shuffle=True)

print("mini-batches per epoch:", len(batch_loader))

In [ ]:
# Train with early stopping: keep the network that does best on validation.
best_validation_loss = float("inf")
epochs_without_improvement = 0
patience = 5          # stop if validation has not improved for 5 epochs

for epoch in range(60):
    # --- learn from every mini-batch (dropout is ON while training) ---
    robust_model.train()
    for batch_inputs, batch_labels in batch_loader:
        batch_scores = robust_model(batch_inputs)
        batch_loss = robust_loss(batch_scores, batch_labels)
        robust_optimizer.zero_grad()
        batch_loss.backward()
        robust_optimizer.step()

    # --- check the validation loss (dropout OFF, no gradients) ---
    robust_model.eval()
    with torch.no_grad():
        validation_scores = robust_model(X_val_t)
        validation_loss = robust_loss(validation_scores, y_val_t).item()

    # --- early-stopping bookkeeping ---
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("stopping early at epoch", epoch + 1)
        break

print("best validation loss:", round(best_validation_loss, 4))

## What you just did

You built a small neural network in PyTorch, trained it with Adam by rolling
downhill, watched its loss fall, and measured how often it reads digits it never
saw. That is a complete, working classifier, and your proof that you trained a
neural network end to end. In the Stretch cells you also met the professional
touches: mini-batches, dropout, and early stopping.

Next notebook: `03_a_network_that_reads_pictures.ipynb`, where we keep each image
as a grid instead of flattening it, and meet the network family behind modern
image recognition.